In [1]:
!nvidia-smi

Tue Aug 18 11:43:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys

print("Python version:")
print(sys.version)


Python version:
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
CUDA available: True


In [4]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
else:
    print("CUDA is NOT available")

GPU: Tesla T4
CUDA version: 12.8
GPU count: 1


In [5]:
if torch.cuda.is_available():
    gpu_properties = torch.cuda.get_device_properties(0)

    print("GPU:", gpu_properties.name)
    print("Total VRAM:",
          round(gpu_properties.total_memory / 1024**3, 2),
          "GB")

GPU: Tesla T4
Total VRAM: 14.56 GB


In [6]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")

    x = torch.randn(2000, 2000, device=device)
    y = torch.randn(2000, 2000, device=device)

    z = torch.matmul(x, y)

    print("GPU computation successful!")
    print("Device:", z.device)
else:
    print("CUDA is not available.")

GPU computation successful!
Device: cuda:0


In [7]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  42% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
tmpfs           6.4G   72K  6.4G   1% /var/colab
/dev/sda1       119G   52G   67G  44% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [8]:
import numpy
import pandas

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)

NumPy: 2.0.2
Pandas: 2.2.2


In [9]:
import platform
import torch
import sys

print("=" * 50)
print("ENVIRONMENT REPORT")
print("=" * 50)

print("Python:", sys.version)
print("OS:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

    props = torch.cuda.get_device_properties(0)
    print("VRAM:",
          round(props.total_memory / 1024**3, 2),
          "GB")

print("=" * 50)

ENVIRONMENT REPORT
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA: 12.8
VRAM: 14.56 GB


In [10]:
from datasets import load_dataset

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

print(dataset)

README.md:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

IterableDataset({
    features: ['audio', 'text', 'gender'],
    num_shards: 4
})


In [11]:
sample = next(iter(dataset))

print(sample.keys())
print(sample)

dict_keys(['audio', 'text', 'gender'])
{'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7a1e7323f500>, 'text': 'त्यांना एक सुंदर स्त्री राजवाड्यातून बाहेर पडतांना दिसली.', 'gender': 0}


In [14]:
sample = next(iter(dataset))

audio = sample["audio"]

print("Audio object:")
print(audio)

print("\nAudio object type:")
print(type(audio))

print("\nGender:")
print(sample["gender"])

print("\nAudio information:")
print(sample["audio"])

Audio object:

Audio object type:
<class 'datasets.features._torchcodec.AudioDecoder'>

Gender:
0

Audio information:


In [15]:
samples = audio.get_all_samples()

print("Sample information:")
print(samples)

print("\nData shape:")
print(samples.data.shape)

print("\nSample rate:")
print(samples.sample_rate)

Sample information:
AudioSamples:
  data (shape): torch.Size([1, 271153])
  pts_seconds: 0.0
  duration_seconds: 5.649020833333333
  sample_rate: 48000


Data shape:
torch.Size([1, 271153])

Sample rate:
48000


In [16]:
import torch

audio = sample["audio"]
samples = audio.get_all_samples()

waveform = samples.data
sample_rate = samples.sample_rate

print("Text:", sample["text"])
print("Gender:", sample["gender"])
print("Sample rate:", sample_rate)
print("Waveform shape:", waveform.shape)
print("Duration:", waveform.shape[-1] / sample_rate, "seconds")
print("Data type:", waveform.dtype)

Text: त्यांना एक सुंदर स्त्री राजवाड्यातून बाहेर पडतांना दिसली.
Gender: 0
Sample rate: 48000
Waveform shape: torch.Size([1, 271153])
Duration: 5.649020833333333 seconds
Data type: torch.float32


In [17]:
from IPython.display import Audio, display

audio_array = waveform.squeeze().cpu().numpy()

display(
    Audio(
        audio_array,
        rate=sample_rate
    )
)

In [18]:
from collections import Counter

gender_counts = Counter()

for i, item in enumerate(dataset):
    gender_counts[item["gender"]] += 1

    if i >= 999:
        break

print(gender_counts)

Counter({0: 1000})


###
inspect the entire dataset's gender distribution

In [19]:
from collections import Counter

gender_counts = Counter()

for i, item in enumerate(dataset):
    gender_counts[item["gender"]] += 1

print("Gender distribution:")
print(gender_counts)

print("\nTotal samples:")
print(sum(gender_counts.values()))

Gender distribution:
Counter({1: 5577, 0: 5362})

Total samples:
10939


Results :
Marathi IndicTTS

────────────────────────

Total samples : 10,939

Gender 0      : 5,362

Gender 1      : 5,577

Audio         : WAV

Sample rate   : 48 kHz


In [20]:
from collections import Counter
import numpy as np

durations = []
text_lengths = []
sample_rates = Counter()
missing_text = 0

for i, item in enumerate(dataset):
    # Text
    text = item["text"]

    if text is None or not str(text).strip():
        missing_text += 1
        text_lengths.append(0)
    else:
        text_lengths.append(len(str(text).strip()))

    # Audio
    audio = item["audio"]
    samples = audio.get_all_samples()

    waveform = samples.data
    sample_rate = samples.sample_rate

    duration = waveform.shape[-1] / sample_rate
    durations.append(duration)

    sample_rates[sample_rate] += 1

print("Total samples:", len(durations))

print("\n--- Audio Statistics ---")
print("Minimum duration:", min(durations), "seconds")
print("Maximum duration:", max(durations), "seconds")
print("Average duration:", np.mean(durations), "seconds")
print("Median duration:", np.median(durations), "seconds")

print("\n--- Transcript Statistics ---")
print("Minimum text length:", min(text_lengths), "characters")
print("Maximum text length:", max(text_lengths), "characters")
print("Average text length:", np.mean(text_lengths), "characters")
print("Median text length:", np.median(text_lengths), "characters")
print("Missing transcripts:", missing_text)

print("\n--- Sampling Rates ---")
print(sample_rates)

Total samples: 10939

--- Audio Statistics ---
Minimum duration: 2.119083333333333 seconds
Maximum duration: 25.825958333333332 seconds
Average duration: 7.362709656961331 seconds
Median duration: 7.014958333333333 seconds

--- Transcript Statistics ---
Minimum text length: 18 characters
Maximum text length: 172 characters
Average text length: 67.19974403510376 characters
Median text length: 64.0 characters
Missing transcripts: 0

--- Sampling Rates ---
Counter({48000: 10939})


In [21]:
from datasets import load_dataset

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

In [22]:
samples_to_check = [0, 1000, 3000, 5361, 5362, 7000, 9000, 10938]

for index, item in enumerate(dataset):
    if index in samples_to_check:
        print(
            f"Index: {index} | "
            f"Gender: {item['gender']} | "
            f"Text: {item['text'][:80]}"
        )

    if index >= max(samples_to_check):
        break

Index: 0 | Gender: 0 | Text: त्यांना एक सुंदर स्त्री राजवाड्यातून बाहेर पडतांना दिसली.
Index: 1000 | Gender: 0 | Text: देवाने त्याच्या जीवनाचे वस्त्र जणू दु:खाने विणले होते.
Index: 3000 | Gender: 0 | Text: त्यामुळे वापरात नव्हते म्हणून थोडे फिक्के पडले होते.
Index: 5361 | Gender: 1 | Text: निमाला जरा धास्ती वाटत होती, पण ती रसरशीत केळी पाहून त्याला मोह आवरेना.
Index: 5362 | Gender: 1 | Text: पिंजरा घेऊन तो डोंगरापलीकडच्या गुहेत आपल्या घरी गेला.
Index: 7000 | Gender: 1 | Text: विदेशात काही मिनिटांच्या रिऍलिटी डॉक्‍युमेंटरीज खूप प्रसिद्ध आहेत.
Index: 9000 | Gender: 1 | Text: ती बातमी ऐकून, शेतक-याला मोठा अचंबा वाटला, तो स्वत:शीच म्हणाला, 'चला, आल्या पावल
Index: 10938 | Gender: 0 | Text: या क्षणी प्रत्यक्ष माझ्या सावलीत बसलेला आहेस, तरीसुध्दा तू मला निरूपयोगी म्हणून 


In [23]:
dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

print(dataset.features)

{'audio': Audio(sampling_rate=None, decode=True, stream_index=None), 'text': Value('string'), 'gender': ClassLabel(names=['female', 'male'])}


In [24]:
print(dataset.features["audio"])
print(dataset.features["text"])
print(dataset.features["gender"])

Audio(sampling_rate=None, decode=True, stream_index=None)
Value('string')
ClassLabel(names=['female', 'male'])


In [25]:
sample = next(iter(dataset))

audio = sample["audio"]

print(type(audio))
print(dir(audio))

<class 'datasets.features._torchcodec.AudioDecoder'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_decoder', '_desired_sample_rate', '_hf_encoded', 'get_all_samples', 'get_samples_played_in_range', 'metadata', 'stream_index']


In [26]:
from datasets import load_dataset

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

valid_audio = 0
corrupted_audio = 0
errors = []

for i, item in enumerate(dataset):
    try:
        samples = item["audio"].get_all_samples()

        if samples.data.numel() == 0:
            corrupted_audio += 1
            errors.append((i, "Empty audio"))
        else:
            valid_audio += 1

    except Exception as e:
        corrupted_audio += 1
        errors.append((i, str(e)))

print("Valid audio:", valid_audio)
print("Corrupted/invalid audio:", corrupted_audio)

if errors:
    print("\nFirst errors:")
    for error in errors[:10]:
        print(error)

Valid audio: 10939
Corrupted/invalid audio: 0


In [27]:
from datasets import load_dataset

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

empty_text = 0
non_string_text = 0
whitespace_text = 0

for i, item in enumerate(dataset):
    text = item["text"]

    if text is None:
        empty_text += 1

    elif not isinstance(text, str):
        non_string_text += 1

    elif not text.strip():
        whitespace_text += 1

print("Empty transcripts:", empty_text)
print("Non-string transcripts:", non_string_text)
print("Whitespace-only transcripts:", whitespace_text)

Empty transcripts: 0
Non-string transcripts: 0
Whitespace-only transcripts: 0


In [28]:
import unicodedata
from collections import Counter

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

char_counts = Counter()

for item in dataset:
    text = item["text"]

    if isinstance(text, str):
        for char in text:
            char_counts[char] += 1

print("Unique characters:", len(char_counts))

print("\nMost common characters:")
for char, count in char_counts.most_common(30):
    print(repr(char), count)

Unique characters: 113

Most common characters:
' ' 102049
'ा' 83785
'्' 40575
'त' 35999
'र' 31873
'े' 31324
'ी' 25893
'य' 24859
'ल' 24660
'न' 21832
'क' 20738
'व' 20175
'स' 17951
'ह' 17087
'म' 16610
'च' 15686
'ं' 14741
'ि' 14238
'प' 14022
'ो' 12300
'ण' 10737
'.' 9889
'द' 9127
'ग' 8569
'आ' 8428
'ु' 8384
'ज' 8017
'ू' 7742
',' 6396
'श' 6146


In [29]:
import unicodedata
from datasets import load_dataset

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

normalization_issues = []

for i, item in enumerate(dataset):
    text = item["text"]

    if isinstance(text, str):
        normalized = unicodedata.normalize("NFC", text)

        if text != normalized:
            normalization_issues.append((i, text, normalized))

print("Total normalization differences:", len(normalization_issues))

if normalization_issues:
    print("\nFirst 5 examples:")
    for item in normalization_issues[:5]:
        print(item)

Total normalization differences: 17

First 5 examples:
(769, 'गार गार सावली, गोड गोड फळे, देवाला फुले, झाडे फ़क़्त एवढेच नाही देत, तर झाडामुळे पाउस पडतो.', 'गार गार सावली, गोड गोड फळे, देवाला फुले, झाडे फ़क़्त एवढेच नाही देत, तर झाडामुळे पाउस पडतो.')
(2369, '‘मी आज सकाळी डेन्टिस्ट कड़े गेलो होतो', '‘मी आज सकाळी डेन्टिस्ट कड़े गेलो होतो')
(2372, 'माझी बायको रोज़ सकाळी मृत्युलेखाचा कॉलम वाचते.', 'माझी बायको रोज़ सकाळी मृत्युलेखाचा कॉलम वाचते.')
(2457, 'विनायकाचा लग्नसोहळा, मोठ्य़ा वैभवाने व थाटात साजरा करण्यात आला.', 'विनायकाचा लग्नसोहळा, मोठ्य़ा वैभवाने व थाटात साजरा करण्यात आला.')
(2748, 'तेवढ्य़ांत त्यांना शेजारच्या खोलीतून कोणाचेतरी ओरडणे ऐकूं आले.', 'तेवढ्य़ांत त्यांना शेजारच्या खोलीतून कोणाचेतरी ओरडणे ऐकूं आले.')


In [30]:
from datasets import load_dataset

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

sample = next(iter(dataset))

audio = sample["audio"]

print("Audio metadata:")
print(audio.metadata)

Audio metadata:
AudioStreamMetadata:
  duration_seconds_from_header: 5.649020833333333
  begin_stream_seconds_from_header: None
  bit_rate: 768000
  codec: pcm_s16le
  stream_index: 0
  duration_seconds: 5.649020833333333
  begin_stream_seconds: 0
  sample_rate: 48000
  num_channels: 1
  sample_format: s16



In [31]:
print("Metadata type:")
print(type(audio.metadata))

Metadata type:
<class 'torchcodec._core._metadata.AudioStreamMetadata'>


In [32]:
print("Encoded audio information:")
print(audio._hf_encoded)

Encoded audio information:
{'path': 'train_marathifemale_00001.wav', 'bytes': b'RIFF\x86F\x08\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80\xbb\x00\x00\x00w\x01\x00\x02\x00\x10\x00databF\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00

In [33]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

paths = []

for i, item in enumerate(dataset):
    path = item["audio"]._hf_encoded["path"]
    paths.append(path)

print("Total paths:", len(paths))

print("\nFirst 20 paths:")
for path in paths[:20]:
    print(path)

Total paths: 10939

First 20 paths:
train_marathifemale_00001.wav
train_marathifemale_00002.wav
train_marathifemale_00003.wav
train_marathifemale_00004.wav
train_marathifemale_00005.wav
train_marathifemale_00006.wav
train_marathifemale_00007.wav
train_marathifemale_00008.wav
train_marathifemale_00009.wav
train_marathifemale_00010.wav
train_marathifemale_00011.wav
train_marathifemale_00012.wav
train_marathifemale_00013.wav
train_marathifemale_00014.wav
train_marathifemale_00015.wav
train_marathifemale_00016.wav
train_marathifemale_00017.wav
train_marathifemale_00018.wav
train_marathifemale_00019.wav
train_marathifemale_00020.wav


In [34]:
from collections import Counter
import os

prefixes = Counter()

for path in paths:
    filename = os.path.basename(path)

    # Remove numeric part, e.g.
    # train_marathifemale_00001.wav
    # → train_marathifemale
    prefix = filename.rsplit("_", 1)[0]

    prefixes[prefix] += 1

print("Filename groups:")
for prefix, count in prefixes.items():
    print(prefix, ":", count)

Filename groups:
train_marathifemale : 5362
text0001.wav : 1
text0002.wav : 1
text0003.wav : 1
text0004.wav : 1
text0005.wav : 1
text0006.wav : 1
text0007.wav : 1
text0008.wav : 1
text0009.wav : 1
text0010.wav : 1
text0011.wav : 1
text0012.wav : 1
text0013.wav : 1
text0014.wav : 1
text0015.wav : 1
text0016.wav : 1
text0017.wav : 1
text0018.wav : 1
text0019.wav : 1
text0020.wav : 1
text0021.wav : 1
text0022.wav : 1
text0023.wav : 1
text0024.wav : 1
text0025.wav : 1
text0026.wav : 1
text0027.wav : 1
text0028.wav : 1
text0029.wav : 1
text0030.wav : 1
text0031.wav : 1
text0032.wav : 1
text0033.wav : 1
text0034.wav : 1
text0035.wav : 1
text0036.wav : 1
text0037.wav : 1
text0038.wav : 1
text0039.wav : 1
text0040.wav : 1
text0041.wav : 1
text0042.wav : 1
text0043.wav : 1
text0044.wav : 1
text0045.wav : 1
text0046.wav : 1
text0047.wav : 1
text0048.wav : 1
text0049.wav : 1
text0050.wav : 1
text0051.wav : 1
text0052.wav : 1
text0053.wav : 1
text0054.wav : 1
text0055.wav : 1
text0056.wav : 1
text

In [35]:
# Show filenames around the point where the naming pattern changes

for i in range(5340, 5390):
    print(i, paths[i])

5340 text0730.wav
5341 text0731.wav
5342 text0732.wav
5343 text0733.wav
5344 text0734.wav
5345 text0735.wav
5346 text0736.wav
5347 text0737.wav
5348 text0738.wav
5349 text0739.wav
5350 text0740.wav
5351 text0741.wav
5352 text0742.wav
5353 text0743.wav
5354 text0744.wav
5355 text0745.wav
5356 text0746.wav
5357 text0747.wav
5358 text0748.wav
5359 text0749.wav
5360 text0750.wav
5361 text0751.wav
5362 text0752.wav
5363 text0753.wav
5364 text0754.wav
5365 text0755.wav
5366 text0756.wav
5367 text0757.wav
5368 text0758.wav
5369 text0759.wav
5370 text0760.wav
5371 text0761.wav
5372 text0762.wav
5373 text0763.wav
5374 text0764.wav
5375 text0765.wav
5376 text0766.wav
5377 text0767.wav
5378 text0768.wav
5379 text0769.wav
5380 text0770.wav
5381 text0771.wav
5382 text0772.wav
5383 text0773.wav
5384 text0774.wav
5385 text0775.wav
5386 text0776.wav
5387 text0777.wav
5388 text0778.wav
5389 text0779.wav


In [36]:
print("\nLast 20 filenames:")

for path in paths[-20:]:
    print(path)


Last 20 filenames:
train_marathifemale_01741.wav
train_marathifemale_01742.wav
train_marathifemale_01743.wav
train_marathifemale_01744.wav
train_marathifemale_01745.wav
train_marathifemale_01746.wav
train_marathifemale_01747.wav
train_marathifemale_01748.wav
train_marathifemale_01749.wav
train_marathifemale_01750.wav
train_marathifemale_01751.wav
train_marathifemale_01752.wav
train_marathifemale_01753.wav
train_marathifemale_01754.wav
train_marathifemale_01755.wav
train_marathifemale_01756.wav
train_marathifemale_01757.wav
train_marathifemale_01758.wav
train_marathifemale_01759.wav
train_marathifemale_01760.wav


In [37]:
from collections import Counter
import os

extensions = Counter(
    os.path.splitext(path)[1].lower()
    for path in paths
)

print("File extensions:")
print(extensions)

File extensions:
Counter({'.wav': 10939})


In [38]:
for i in range(5340, 5390):
    print(i, paths[i])

5340 text0730.wav
5341 text0731.wav
5342 text0732.wav
5343 text0733.wav
5344 text0734.wav
5345 text0735.wav
5346 text0736.wav
5347 text0737.wav
5348 text0738.wav
5349 text0739.wav
5350 text0740.wav
5351 text0741.wav
5352 text0742.wav
5353 text0743.wav
5354 text0744.wav
5355 text0745.wav
5356 text0746.wav
5357 text0747.wav
5358 text0748.wav
5359 text0749.wav
5360 text0750.wav
5361 text0751.wav
5362 text0752.wav
5363 text0753.wav
5364 text0754.wav
5365 text0755.wav
5366 text0756.wav
5367 text0757.wav
5368 text0758.wav
5369 text0759.wav
5370 text0760.wav
5371 text0761.wav
5372 text0762.wav
5373 text0763.wav
5374 text0764.wav
5375 text0765.wav
5376 text0766.wav
5377 text0767.wav
5378 text0768.wav
5379 text0769.wav
5380 text0770.wav
5381 text0771.wav
5382 text0772.wav
5383 text0773.wav
5384 text0774.wav
5385 text0775.wav
5386 text0776.wav
5387 text0777.wav
5388 text0778.wav
5389 text0779.wav


In [39]:
print("Total paths:", len(paths))
print("Unique paths:", len(set(paths)))
print("Duplicate paths:", len(paths) - len(set(paths)))

Total paths: 10939
Unique paths: 10188
Duplicate paths: 751


In [40]:
from datasets import load_dataset
from collections import Counter

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

filename_gender = Counter()
mismatches = []

for i, item in enumerate(dataset):
    path = item["audio"]._hf_encoded["path"]
    gender = item["gender"]

    filename_lower = path.lower()

    if "female" in filename_lower:
        filename_label = 0
    elif "male" in filename_lower:
        filename_label = 1
    else:
        filename_label = None

    filename_gender[(gender, filename_label)] += 1

    if filename_label is not None and gender != filename_label:
        mismatches.append((i, path, gender, filename_label))

print("Gender/filename combinations:")
for key, count in filename_gender.items():
    print(key, ":", count)

print("\nMismatches:", len(mismatches))

if mismatches:
    print("\nFirst mismatches:")
    for x in mismatches[:10]:
        print(x)

Gender/filename combinations:
(0, 0) : 5362
(1, None) : 4766
(1, 1) : 811

Mismatches: 0


In [41]:
from collections import Counter

path_counts = Counter(paths)

duplicate_paths = {
    path: count
    for path, count in path_counts.items()
    if count > 1
}

print("Number of duplicated path names:", len(duplicate_paths))
print("Total duplicate occurrences:",
      sum(count - 1 for count in duplicate_paths.values()))

print("\nFirst 30 duplicated paths:")

for path, count in list(duplicate_paths.items())[:30]:
    print(path, "->", count)

Number of duplicated path names: 751
Total duplicate occurrences: 751

First 30 duplicated paths:
train_marathifemale_00955.wav -> 2
train_marathifemale_00956.wav -> 2
train_marathifemale_00957.wav -> 2
train_marathifemale_00958.wav -> 2
train_marathifemale_00959.wav -> 2
train_marathifemale_00960.wav -> 2
train_marathifemale_00961.wav -> 2
train_marathifemale_00962.wav -> 2
train_marathifemale_00963.wav -> 2
train_marathifemale_00964.wav -> 2
train_marathifemale_00965.wav -> 2
train_marathifemale_00966.wav -> 2
train_marathifemale_00967.wav -> 2
train_marathifemale_00968.wav -> 2
train_marathifemale_00969.wav -> 2
train_marathifemale_00970.wav -> 2
train_marathifemale_00971.wav -> 2
train_marathifemale_00972.wav -> 2
train_marathifemale_00973.wav -> 2
train_marathifemale_00974.wav -> 2
train_marathifemale_00975.wav -> 2
train_marathifemale_00976.wav -> 2
train_marathifemale_00977.wav -> 2
train_marathifemale_00978.wav -> 2
train_marathifemale_00979.wav -> 2
train_marathifemale_00980.w

In [42]:
from collections import Counter

duplicate_frequency = Counter(path_counts.values())

print("Path frequency distribution:")

for frequency, number_of_paths in sorted(duplicate_frequency.items()):
    print(
        f"{frequency} occurrence(s): "
        f"{number_of_paths} path(s)"
    )

Path frequency distribution:
1 occurrence(s): 9437 path(s)
2 occurrence(s): 751 path(s)


In [43]:
from datasets import load_dataset

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Marathi",
    split="train",
    streaming=True
)

target_path = "train_marathifemale_00955.wav"

matches = []

for i, item in enumerate(dataset):
    path = item["audio"]._hf_encoded["path"]

    if path == target_path:
        audio_samples = item["audio"].get_all_samples()

        matches.append({
            "index": i,
            "path": path,
            "gender": item["gender"],
            "text": item["text"],
            "sample_rate": audio_samples.sample_rate,
            "shape": tuple(audio_samples.data.shape),
            "duration": audio_samples.data.shape[-1] / audio_samples.sample_rate,
            "audio": audio_samples.data.clone()
        })

print("Number of occurrences:", len(matches))

for m in matches:
    print("\nIndex:", m["index"])
    print("Gender:", m["gender"])
    print("Text:", m["text"])
    print("Sample rate:", m["sample_rate"])
    print("Shape:", m["shape"])
    print("Duration:", m["duration"])

Number of occurrences: 2

Index: 954
Gender: 0
Text: वेळेत न जेवल्यास शरीरस्वास्थ्य बिघडते.
Sample rate: 48000
Shape: (1, 212274)
Duration: 4.422375

Index: 10188
Gender: 0
Text: वेळेत न जेवल्यास शरीरस्वास्थ्य बिघडते.
Sample rate: 48000
Shape: (1, 183034)
Duration: 3.8132083333333333


In [44]:
import torch

if len(matches) == 2:
    audio1 = matches[0]["audio"]
    audio2 = matches[1]["audio"]

    print("Same shape:", audio1.shape == audio2.shape)

    if audio1.shape == audio2.shape:
        print(
            "Exactly identical audio:",
            torch.equal(audio1, audio2)
        )

        print(
            "Maximum absolute difference:",
            torch.max(torch.abs(audio1 - audio2)).item()
        )

Same shape: False


In [45]:
for i, m in enumerate(matches):
    print(f"\nOccurrence {i + 1}")
    print("Index:", m["index"])
    print("Path:", m["path"])
    print("Gender:", m["gender"])
    print("Text:", m["text"])
    print("Sample rate:", m["sample_rate"])
    print("Shape:", m["shape"])
    print("Duration:", m["duration"])


Occurrence 1
Index: 954
Path: train_marathifemale_00955.wav
Gender: 0
Text: वेळेत न जेवल्यास शरीरस्वास्थ्य बिघडते.
Sample rate: 48000
Shape: (1, 212274)
Duration: 4.422375

Occurrence 2
Index: 10188
Path: train_marathifemale_00955.wav
Gender: 0
Text: वेळेत न जेवल्यास शरीरस्वास्थ्य बिघडते.
Sample rate: 48000
Shape: (1, 183034)
Duration: 3.8132083333333333


In [46]:
audio1 = matches[0]["audio"]
audio2 = matches[1]["audio"]

print("Shape 1:", audio1.shape)
print("Shape 2:", audio2.shape)

# Compare only if shapes are equal
if audio1.shape == audio2.shape:
    print("Identical:", torch.equal(audio1, audio2))
else:
    print("Audio lengths differ, so they are different recordings.")

Shape 1: torch.Size([1, 212274])
Shape 2: torch.Size([1, 183034])
Audio lengths differ, so they are different recordings.


In [47]:
from pathlib import Path

BASE_DIR = Path("/content/Gyan")

directories = [
    BASE_DIR / "data" / "processed" / "english" / "wavs",
    BASE_DIR / "data" / "processed" / "hindi" / "wavs",
    BASE_DIR / "data" / "processed" / "marathi" / "wavs",
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Processed dataset directories created.")

Processed dataset directories created.


In [48]:
for directory in directories:
    print(directory)

/content/Gyan/data/processed/english/wavs
/content/Gyan/data/processed/hindi/wavs
/content/Gyan/data/processed/marathi/wavs
